In [1]:

# ════════════════════════════════════════════════════════════════════════════
# FILE 1 — CLASSIFIER TRAINING  (v2 — UNIFIED SPLIT + 3-BRANCH META-ENSEMBLE)
# Trains: Morphological Stacked Ensemble + VGG16 + EfficientNetV2L
# Fuses : 0.20 * Morph  +  0.40 * VGG16  +  0.40 * EfficientNetV2L
# Saves : all model files + probability tensors + CSV + plots into OUTPUT_DIR
# Next  : run file2_train_ood.py
#
# CHANGES vs v1 (see chat notes):
#   [C1] Single unified train/val/test split shared by ALL three branches,
#        keyed on (canonical_class, filename_stem) composite identity.
#   [C2] Canonical class indexing for the CNN branch (no folder-order drift).
#   [C3] Imputer / outlier clip / scaler now fit on TRAIN ONLY (leakage fix).
#   [C4] Masks written into per-class subdirectories (collision fix).
#   [C5] NEW Cell 13 — three-branch weighted meta-ensemble + weight sweep.
# ════════════════════════════════════════════════════════════════════════════

# ── IMPORTS ──────────────────────────────────────────────────────────────────
import os, cv2, math, glob, zipfile, warnings, pickle, json, itertools, time
from collections import Counter
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                        ReduceLROnPlateau)

from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing     import LabelEncoder, RobustScaler
from sklearn.model_selection   import StratifiedKFold, train_test_split
from sklearn.impute            import SimpleImputer

# -- Standard & Advanced ML Algorithms --
from sklearn.ensemble          import (RandomForestClassifier, ExtraTreesClassifier,
                                       HistGradientBoostingClassifier, StackingClassifier,
                                       AdaBoostClassifier, GradientBoostingClassifier)
from sklearn.svm               import SVC
from sklearn.neural_network    import MLPClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

from sklearn.metrics           import (accuracy_score, classification_report,
                                       confusion_matrix, cohen_kappa_score,
                                       balanced_accuracy_score)
from sklearn.calibration       import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
from IPython.display           import FileLink, display

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
BMP_DIR    = '/kaggle/input/datasets/sazedkarim/bmp-with-morph/BMP Data'
PNG_DIR    = '/kaggle/input/datasets/sazedkarim/pap-png/png'
OUTPUT_DIR = 'sipakmed_unified_output'

IMAGE_SIZE        = (224, 224)
NUM_CLASSES       = 5
SEED              = 1337
TRAIN_BATCH_SIZE  = 16
TTA_BATCH_SIZE    = 8
TTA_STEPS         = 5

# [C1] ─── ONE split for every branch. These four constants replaced
#          VALIDATION_SPLIT / TEST_SPLIT / VAL_SIZE_MORPH / TEST_SIZE_MORPH.
#          A single TEST_SIZE is what makes the meta-ensemble evaluable.
TEST_SIZE         = 0.15   # held-out, never touched until final evaluation
VAL_SIZE          = 0.15   # model selection / early stopping / weight tuning

GLCM_DISTANCES    = [1, 3]
GLCM_ANGLES       = [0, np.pi/4, np.pi/2, 3*np.pi/4]

# Meta-ensemble fusion weights (must sum to 1.0)
WEIGHT_MORPH      = 0.20
WEIGHT_VGG        = 0.40
WEIGHT_EFF        = 0.40
assert abs(WEIGHT_MORPH + WEIGHT_VGG + WEIGHT_EFF - 1.0) < 1e-9

CANONICAL_CLASSES = [
    'Dyskeratotic', 'Koilocytotic', 'Metaplastic',
    'Parabasal', 'Superficial-Intermediate'
]
# Canonical, alphabetical, immutable class->index map used by EVERY branch.
CLASS_TO_ID = {c: i for i, c in enumerate(CANONICAL_CLASSES)}

for d in [OUTPUT_DIR,
          f'{OUTPUT_DIR}/masks/nucleus',
          f'{OUTPUT_DIR}/masks/cytoplasm',
          f'{OUTPUT_DIR}/plots',
          f'{OUTPUT_DIR}/models',
          f'{OUTPUT_DIR}/probs',
          f'{OUTPUT_DIR}/reports']:
    os.makedirs(d, exist_ok=True)
# [C4] per-class mask subdirectories — SIPaKMeD reuses filenames across classes
for cls in CANONICAL_CLASSES:
    os.makedirs(f'{OUTPUT_DIR}/masks/nucleus/{cls}',   exist_ok=True)
    os.makedirs(f'{OUTPUT_DIR}/masks/cytoplasm/{cls}', exist_ok=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)
print(f'TensorFlow : {tf.__version__}')
print(f'GPUs       : {tf.config.list_physical_devices("GPU")}')
print(f'Split      : test={TEST_SIZE}  val={VAL_SIZE}  seed={SEED}  (UNIFIED)')
print(f'Fusion     : morph={WEIGHT_MORPH}  vgg={WEIGHT_VGG}  eff={WEIGHT_EFF}')
print('✅ Configuration complete.')

TensorFlow : 2.20.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Split      : test=0.15  val=0.15  seed=1337  (UNIFIED)
Fusion     : morph=0.2  vgg=0.4  eff=0.4
✅ Configuration complete.


In [2]:

# ════════════════════════════════════════════════════════════
# CELL 1-2 — SEGMENTATION & FEATURE EXTRACTION
# [C6 / NEW in v3] GROUND-TRUTH-FIRST SEGMENTATION WITH OTSU FALLBACK
#
# WHY THIS CHANGED
# ----------------
# v2 used Otsu + distance-transform for EVERY image. SIPaKMeD ships expert
# contour annotations (*_nuc*.dat / *_cyt*.dat) alongside the BMPs, and the
# thesis Methodology (§3.3.1) claims a GT-first pipeline. Running Otsu on
# images that HAVE ground truth discards expert annotation and makes every
# morphological descriptor heuristic. This cell restores the documented
# behaviour and — critically — RECORDS THE PROVENANCE of every mask so the
# contribution of heuristic segmentation to error can be measured (§4.2.2),
# not assumed negligible.
#
# Provenance codes:  'GT'   both masks from expert contours
#                    'OTSU' fallback used (no sidecar, or degenerate GT)
# ════════════════════════════════════════════════════════════

# ── IDENTITY HELPERS (restored — these were defined in this cell in v2) ──────
def _canonical_name(folder_name):
    """'4_im_Superficial-Intermediate' -> 'Superficial-Intermediate'"""
    parts = folder_name.split('_im_')
    return parts[-1] if len(parts) > 1 else folder_name

def _stem(path_or_name):
    """Filename without directory or extension. '025_01.png' -> '025_01'"""
    return os.path.splitext(os.path.basename(path_or_name))[0]


def _read_dat_contour(path):
    """Parse a SIPaKMeD .dat contour file into an (N,2) int array."""
    try:
        arr = np.loadtxt(path, delimiter=',')
    except Exception:
        try:
            arr = np.loadtxt(path)
        except Exception:
            return None
    arr = np.atleast_2d(arr)
    if arr.ndim != 2 or arr.shape[1] < 2 or arr.shape[0] < 3:
        return None
    return np.round(arr[:, :2]).astype(np.int32)


def _rasterise(contours, shape):
    """Fill a list of polygons into a uint8 {0,255} mask."""
    mask = np.zeros(shape, dtype=np.uint8)
    filled = 0
    for c in contours:
        if c is None:
            continue
        cv2.fillPoly(mask, [c.reshape(-1, 1, 2)], 255)
        filled += 1
    return mask if filled else None


def segment_from_groundtruth(image_path, gray_shape):
    """
    Primary tier. SIPaKMeD annotates the cytoplasm contour as the WHOLE-CELL
    boundary, so the nucleus is subtracted to make the two regions disjoint
    (Methodology Eq. 3.2):   M_cyto <- M_cyto AND NOT M_nuc
    """
    base = os.path.splitext(image_path)[0]
    nuc_files = sorted(glob.glob(base + '_nuc*.dat'))
    cyt_files = sorted(glob.glob(base + '_cyt*.dat'))
    if not nuc_files or not cyt_files:
        return None, None

    nuc_mask  = _rasterise([_read_dat_contour(f) for f in nuc_files], gray_shape)
    cell_mask = _rasterise([_read_dat_contour(f) for f in cyt_files], gray_shape)
    if nuc_mask is None or cell_mask is None:
        return None, None

    cyto_mask = cv2.bitwise_and(cell_mask, cv2.bitwise_not(nuc_mask))

    # Degeneracy check — reject GT and fall back if either region is tiny.
    if np.sum(nuc_mask == 255) < 50 or np.sum(cyto_mask == 255) < 50:
        return None, None
    return nuc_mask, cyto_mask


def segment_cell_parts_otsu(image_gray):
    """Fallback tier — inverse-Otsu + morphology + distance-transform marker."""
    blurred   = cv2.GaussianBlur(image_gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opening = cv2.morphologyEx(thresh,  cv2.MORPH_OPEN,  kernel, iterations=2)
    opening = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, iterations=1)
    dist    = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    if dist.max() == 0:
        return None, None
    _, nuc_mask = cv2.threshold(dist, 0.50 * dist.max(), 255, 0)
    nuc_mask    = np.uint8(nuc_mask)
    cyto_mask   = cv2.bitwise_and(opening, cv2.bitwise_not(nuc_mask))
    return nuc_mask, cyto_mask


def segment_cell_parts(image_path):
    """
    Two-tier dispatcher.
    Returns (nuc_mask, cyto_mask, bgr, gray, source) where source in
    {'GT','OTSU',None}.
    """
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        return None, None, None, None, None
    image_gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    nuc, cyto = segment_from_groundtruth(image_path, image_gray.shape)
    if nuc is not None:
        return nuc, cyto, image_bgr, image_gray, 'GT'

    nuc, cyto = segment_cell_parts_otsu(image_gray)
    if nuc is None:
        return None, None, image_bgr, image_gray, None
    return nuc, cyto, image_bgr, image_gray, 'OTSU'


def safe_circularity(area, perimeter):
    if perimeter <= 0: return 0.0
    return float(np.clip((4 * math.pi * area) / (perimeter ** 2), 0.0, 1.0))

def shape_features_from_mask(mask, prefix):
    feats = {}
    area  = float(np.sum(mask == 255))
    feats[f'{prefix}_area']     = area
    feats[f'{prefix}_log_area'] = np.log1p(area)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        for k in ['perimeter','circularity','aspect_ratio','extent',
                  'solidity','equiv_diameter','hu1','hu2','hu3','hu4']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    cnt    = max(contours, key=cv2.contourArea)
    perim  = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)
    hull   = cv2.convexHull(cnt)
    hull_a = cv2.contourArea(hull)
    equiv_d = math.sqrt(4 * area / math.pi) if area > 0 else 0.0
    M  = cv2.moments(cnt)
    hu = cv2.HuMoments(M).flatten()
    hu_l = [-np.sign(h_) * np.log10(abs(h_) + 1e-10) for h_ in hu[:4]]
    feats[f'{prefix}_perimeter']      = np.log1p(perim)
    feats[f'{prefix}_circularity']    = safe_circularity(area, perim)
    feats[f'{prefix}_aspect_ratio']   = float(w) / h if h > 0 else 0.0
    feats[f'{prefix}_extent']         = area / (w * h) if (w * h) > 0 else 0.0
    feats[f'{prefix}_solidity']       = area / hull_a if hull_a > 0 else 0.0
    feats[f'{prefix}_equiv_diameter'] = np.log1p(equiv_d)
    feats[f'{prefix}_hu1']            = hu_l[0]
    feats[f'{prefix}_hu2']            = hu_l[1]
    feats[f'{prefix}_hu3']            = hu_l[2]
    feats[f'{prefix}_hu4']            = hu_l[3]
    return feats

def eccentricity_from_mask(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return 0.0
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 5: return 0.0
    try:
        (_, axes, _) = cv2.fitEllipse(cnt)
        MA, ma = max(axes), min(axes)
        return float(math.sqrt(1 - (ma / MA) ** 2)) if MA > 0 else 0.0
    except Exception:
        return 0.0

def intensity_features(gray, mask, prefix):
    vals = gray[mask == 255]
    if vals.size == 0:
        return {f'{prefix}_intensity_mean': 0.0, f'{prefix}_intensity_std': 0.0}
    return {f'{prefix}_intensity_mean': float(vals.mean()),
            f'{prefix}_intensity_std' : float(vals.std())}

def hsv_features(bgr, mask, prefix):
    hsv  = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    out  = {}
    for ci, cn in enumerate(['H', 'S', 'V']):
        vals = hsv[:, :, ci][mask == 255]
        out[f'{prefix}_hsv_{cn}_mean'] = float(vals.mean()) if vals.size else 0.0
    return out

def glcm_features(gray, nuc_mask):
    ys, xs = np.where(nuc_mask == 255)
    if ys.size == 0:
        return {f'glcm_{p}': 0.0 for p in
                ['contrast','dissimilarity','homogeneity','energy','correlation']}
    patch = gray[ys.min():ys.max()+1, xs.min():xs.max()+1]
    if patch.size == 0 or min(patch.shape) < 4:
        return {f'glcm_{p}': 0.0 for p in
                ['contrast','dissimilarity','homogeneity','energy','correlation']}
    glcm = graycomatrix(patch, distances=[1, 3],
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    return {f'glcm_{p}': float(graycoprops(glcm, p).mean())
            for p in ['contrast','dissimilarity','homogeneity','energy','correlation']}

def extract_all_features(image_path, nuc_mask, cyto_mask, bgr, gray):
    f = {}
    f.update(shape_features_from_mask(nuc_mask,  'nuc'))
    f.update(shape_features_from_mask(cyto_mask, 'cyto'))
    nuc_a  = f['nuc_area']; cyto_a = f['cyto_area']; cell_a = nuc_a + cyto_a
    f['nc_area_ratio']     = nuc_a / cyto_a if cyto_a > 0 else 0.0
    f['nuc_cell_ratio']    = nuc_a / cell_a if cell_a > 0 else 0.0
    f['cyto_cell_ratio']   = cyto_a / cell_a if cell_a > 0 else 0.0
    f['nc_circ_ratio']     = (f['nuc_circularity'] / f['cyto_circularity']
                              if f['cyto_circularity'] > 0 else 0.0)
    f['nc_solidity_ratio'] = (f['nuc_solidity'] / f['cyto_solidity']
                              if f['cyto_solidity'] > 0 else 0.0)
    f['nuc_eccentricity']  = eccentricity_from_mask(nuc_mask)
    f['cyto_eccentricity'] = eccentricity_from_mask(cyto_mask)
    f.update(intensity_features(gray, nuc_mask,  'nuc'))
    f.update(intensity_features(gray, cyto_mask, 'cyto'))
    f['intensity_contrast'] = f['cyto_intensity_mean'] - f['nuc_intensity_mean']
    f['intensity_ratio']    = (f['nuc_intensity_mean'] / f['cyto_intensity_mean']
                               if f['cyto_intensity_mean'] > 0 else 0.0)
    f.update(hsv_features(bgr, nuc_mask,  'nuc'))
    f.update(hsv_features(bgr, cyto_mask, 'cyto'))
    f.update(glcm_features(gray, nuc_mask))
    return f

print('Feature extraction + two-tier segmentation functions defined.')


Feature extraction + two-tier segmentation functions defined.


In [3]:

# ════════════════════════════════════════════════════════════
# CELL 3 — MORPHOLOGICAL PIPELINE  (BMP -> CSV + masks)
# [C4] masks written to masks/<part>/<canonical_class>/<file>.png
# [C6] seg_source recorded per image  -> enables the §4.9 provenance ablation
# ════════════════════════════════════════════════════════════

def run_morph_pipeline(input_dir, output_root):
    results, failed = [], []
    class_folders = sorted([
        f for f in os.listdir(input_dir)
        if os.path.isdir(os.path.join(input_dir, f))
    ])
    print(f'Found {len(class_folders)} class folders: {class_folders}')
    for class_name in class_folders:
        canon      = _canonical_name(class_name)
        class_path = os.path.join(input_dir, class_name)
        for part in ['nucleus', 'cytoplasm']:
            os.makedirs(f'{output_root}/masks/{part}/{canon}', exist_ok=True)
        images = sorted(glob.glob(os.path.join(class_path, '*.bmp')) +
                        glob.glob(os.path.join(class_path, '*.png')))
        print(f'  {class_name}: {len(images)} images', end='', flush=True)
        ok = Counter()
        for img_path in images:
            fname = os.path.basename(img_path)
            nuc_mask, cyto_mask, bgr, gray, source = segment_cell_parts(img_path)
            if nuc_mask is None or np.sum(nuc_mask == 255) < 50:
                failed.append((canon, fname)); continue
            cv2.imwrite(f'{output_root}/masks/nucleus/{canon}/{_stem(fname)}.png',   nuc_mask)
            cv2.imwrite(f'{output_root}/masks/cytoplasm/{canon}/{_stem(fname)}.png', cyto_mask)
            feat_row               = extract_all_features(img_path, nuc_mask, cyto_mask, bgr, gray)
            feat_row['Image']      = fname
            feat_row['Class']      = canon
            feat_row['seg_source'] = source
            results.append(feat_row)
            ok[source] += 1
        print(f'  -> GT {ok["GT"]} | OTSU {ok["OTSU"]} | failed {len(images)-sum(ok.values())}')
    df       = pd.DataFrame(results)
    csv_path = f'{output_root}/cell_features_enhanced.csv'
    df.to_csv(csv_path, index=False)
    n_feat = len([c for c in df.columns if c not in ['Image','Class','seg_source']])
    print(f'\nMorph pipeline done. {len(results)} images | {n_feat} features')
    print(f'   Segmentation failures : {len(failed)}')
    print(f'   Provenance            : {df["seg_source"].value_counts().to_dict()}')
    return df, csv_path

df_morph, csv_path = run_morph_pipeline(BMP_DIR, OUTPUT_DIR)
print(f'Class distribution:\n{df_morph["Class"].value_counts()}')
assert set(df_morph['Class'].unique()) == set(CANONICAL_CLASSES), \
    'Class names did not canonicalise - check _canonical_name() against folder naming.'

# ── ITEM 2: corpus-wide provenance table (thesis Table 4.1 / §4.2.2) ──────
prov_corpus = (df_morph.groupby(['Class', 'seg_source']).size()
               .unstack(fill_value=0))
prov_corpus['Total'] = prov_corpus.sum(axis=1)
if 'GT' in prov_corpus.columns:
    prov_corpus['GT_%'] = (prov_corpus['GT'] / prov_corpus['Total'] * 100).round(2)
print('\n── Segmentation provenance, whole corpus ──')
print(prov_corpus.to_string())


Found 5 class folders: ['0_im_Dyskeratotic', '1_im_Koilocytotic', '2_im_Metaplastic', '3_im_Parabasal', '4_im_Superficial-Intermediate']
  0_im_Dyskeratotic: 813 images  -> GT 813 | OTSU 0 | failed 0
  1_im_Koilocytotic: 825 images  -> GT 808 | OTSU 17 | failed 0
  2_im_Metaplastic: 793 images  -> GT 792 | OTSU 1 | failed 0
  3_im_Parabasal: 787 images  -> GT 787 | OTSU 0 | failed 0
  4_im_Superficial-Intermediate: 831 images  -> GT 824 | OTSU 7 | failed 0

Morph pipeline done. 4049 images | 48 features
   Segmentation failures : 0
   Provenance            : {'GT': 4024, 'OTSU': 25}
Class distribution:
Class
Superficial-Intermediate    831
Koilocytotic                825
Dyskeratotic                813
Metaplastic                 793
Parabasal                   787
Name: count, dtype: int64

── Segmentation provenance, whole corpus ──
seg_source                 GT  OTSU  Total    GT_%
Class                                             
Dyskeratotic              813     0    813  100.00


In [4]:

# ════════════════════════════════════════════════════════════
# CELL 4 — RAW CNN IMAGE LOAD (paths retained)                [NEW / C1]
# Loaded BEFORE splitting so that the unified registry can be built
# from the intersection of the two modalities.
# ════════════════════════════════════════════════════════════

print(f'Loading PNG dataset from: {PNG_DIR}')
full_ds_unbatched = tf.keras.utils.image_dataset_from_directory(
    PNG_DIR, label_mode='int', image_size=IMAGE_SIZE,
    batch_size=None, seed=SEED, shuffle=False
)
png_folder_names = full_ds_unbatched.class_names
png_file_paths   = list(full_ds_unbatched.file_paths)   # order == iteration order
assert set(_canonical_name(c) for c in png_folder_names) == set(CANONICAL_CLASSES)

all_images_list = []
for img, lbl in full_ds_unbatched:
    all_images_list.append(img.numpy().astype(np.uint8))

# [C1] composite identity key: (canonical_class, filename_stem)
#      the stem alone is NOT unique — SIPaKMeD numbers files per class folder.
png_keys = [(_canonical_name(os.path.basename(os.path.dirname(p))), _stem(p))
            for p in png_file_paths]

assert len(png_keys) == len(all_images_list) == len(png_file_paths)
assert len(set(png_keys)) == len(png_keys), 'Duplicate (class, stem) in PNG tree.'
print(f'PNG images loaded : {len(all_images_list)}')
print(f'Unique stems      : {len(set(k[1] for k in png_keys))}  '
      f'(vs {len(png_keys)} images → composite key required)')

Loading PNG dataset from: /kaggle/input/datasets/sazedkarim/pap-png/png
Found 4049 files belonging to 5 classes.


I0000 00:00:1791301916.152951      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791301916.156749      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


PNG images loaded : 4049
Unique stems      : 1860  (vs 4049 images → composite key required)


In [5]:

# ════════════════════════════════════════════════════════════
# CELL 5 — UNIFIED SAMPLE REGISTRY & SINGLE SPLIT             [NEW / C1]
#
# Scientific rationale
# --------------------
# A weighted meta-ensemble  P_fused = Σ_b w_b · P_b  is only defined when every
# branch b emits a probability vector for the SAME sample i, in the SAME class
# index space. Independently drawn per-branch splits violate both conditions:
# the sample sets differ, and a sample held out by one branch may be a TRAINING
# sample of another — which inflates the fused accuracy through leakage rather
# than through genuine complementarity.
#
# We therefore define ONE registry over the intersection of the two modalities
# and draw ONE stratified split from it. Images present in only one modality
# (i.e. segmentation failures) are excluded from all branches so the row
# correspondence is exact by construction.
# ════════════════════════════════════════════════════════════

morph_key2row = {(r.Class, _stem(r.Image)): i
                 for i, r in enumerate(df_morph.itertuples(index=False))}
png_key2row   = {k: i for i, k in enumerate(png_keys)}

common_keys = sorted(set(morph_key2row) & set(png_key2row))
n_common    = len(common_keys)
print(f'Morph rows        : {len(morph_key2row)}')
print(f'PNG images        : {len(png_key2row)}')
print(f'Unified registry  : {n_common}  '
      f'(dropped {len(morph_key2row)-n_common} morph-only, '
      f'{len(png_key2row)-n_common} png-only)')

# Canonical label for every registry entry — the SINGLE source of truth.
y_unified = np.array([CLASS_TO_ID[k[0]] for k in common_keys], dtype=np.int64)

# ── the one and only split ────────────────────────────────────────────────
reg_idx = np.arange(n_common)
idx_tv, idx_test = train_test_split(
    reg_idx, test_size=TEST_SIZE, random_state=SEED, stratify=y_unified)
_val_ratio_adj = VAL_SIZE / (1.0 - TEST_SIZE)
idx_train, idx_val = train_test_split(
    idx_tv, test_size=_val_ratio_adj, random_state=SEED, stratify=y_unified[idx_tv])

# Registry positions are the shared coordinate system for every branch.
SPLIT = {'train': idx_train, 'val': idx_val, 'test': idx_test}
for name, idx in SPLIT.items():
    dist = Counter(y_unified[idx])
    print(f'{name:>5}: {len(idx):>5}  ' +
          '  '.join(f'{CANONICAL_CLASSES[c][:6]}={dist[c]}' for c in range(NUM_CLASSES)))

# Disjointness guard — cheap, and catches any future edit that breaks the split.
assert len(set(idx_train) & set(idx_val))  == 0
assert len(set(idx_train) & set(idx_test)) == 0
assert len(set(idx_val)   & set(idx_test)) == 0
assert len(idx_train) + len(idx_val) + len(idx_test) == n_common

# Row-index translations into each branch's native storage order
morph_rows = np.array([morph_key2row[k] for k in common_keys])
png_rows   = np.array([png_key2row[k]   for k in common_keys])

test_keys      = [common_keys[i] for i in idx_test]
test_filenames = [f'{c}/{s}' for c, s in test_keys]   # collision-safe identifier

# BMP paths of the TRAINING partition (consumed by file2 OOD calibration)
bmp_train_paths = []
_bmp_folders = [f for f in os.listdir(BMP_DIR) if os.path.isdir(os.path.join(BMP_DIR, f))]
_canon2folder = {_canonical_name(f): f for f in _bmp_folders}
for i in idx_train:
    cls, stem = common_keys[i]
    folder = _canon2folder[cls]
    for ext in ('.bmp', '.png'):
        p = os.path.join(BMP_DIR, folder, stem + ext)
        if os.path.exists(p):
            bmp_train_paths.append(p); break
print(f'\nBMP train paths resolved: {len(bmp_train_paths)} / {len(idx_train)}')
print('✅ Unified split ready — all three branches now share it.')

Morph rows        : 4049
PNG images        : 4049
Unified registry  : 4049  (dropped 0 morph-only, 0 png-only)
train:  2833  Dysker=569  Koiloc=577  Metapl=555  Paraba=551  Superf=581
  val:   608  Dysker=122  Koiloc=124  Metapl=119  Paraba=118  Superf=125
 test:   608  Dysker=122  Koiloc=124  Metapl=119  Paraba=118  Superf=125

BMP train paths resolved: 2833 / 2833
✅ Unified split ready — all three branches now share it.


In [6]:

# ════════════════════════════════════════════════════════════
# CELL 5b — SPLIT COMPOSITION & TEST-SET PROVENANCE   [NEW v3]
# Supplies thesis Table 3.2 (exact per-class counts, not proportional
# estimates) and §4.2.2 (test-set segmentation provenance).
# ════════════════════════════════════════════════════════════

# ── ITEM 1: exact per-class split counts ─────────────────────────────────
split_tbl = pd.DataFrame({
    'Train': pd.Series(y_unified[idx_train]).value_counts().sort_index(),
    'Val'  : pd.Series(y_unified[idx_val]).value_counts().sort_index(),
    'Test' : pd.Series(y_unified[idx_test]).value_counts().sort_index(),
})
split_tbl.index = [CANONICAL_CLASSES[i] for i in split_tbl.index]
split_tbl['Total'] = split_tbl.sum(axis=1)
split_tbl.loc['TOTAL'] = split_tbl.sum()
print('── Table 3.2: unified stratified partition ──')
print(split_tbl.to_string())
split_tbl.to_csv(f'{OUTPUT_DIR}/reports/split_composition.csv')

# ── ITEM 2: provenance restricted to the held-out test set ───────────────
seg_source_registry = df_morph['seg_source'].values[morph_rows]   # registry order
SEG_TEST = seg_source_registry[idx_test]

prov_test = pd.Series(SEG_TEST).value_counts()
print('\n── Segmentation provenance, TEST SET ONLY ──')
for k, v in prov_test.items():
    print(f'   {k:6s} : {v:4d}  ({v/len(SEG_TEST)*100:.2f}%)')

prov_by_class = pd.crosstab(
    pd.Series([CANONICAL_CLASSES[i] for i in y_unified[idx_test]]),
    pd.Series(SEG_TEST))
print('\n── Test-set provenance by class ──')
print(prov_by_class.to_string())
prov_by_class.to_csv(f'{OUTPUT_DIR}/reports/provenance_test.csv')


── Table 3.2: unified stratified partition ──
                          Train  Val  Test  Total
Dyskeratotic                569  122   122    813
Koilocytotic                577  124   124    825
Metaplastic                 555  119   119    793
Parabasal                   551  118   118    787
Superficial-Intermediate    581  125   125    831
TOTAL                      2833  608   608   4049

── Segmentation provenance, TEST SET ONLY ──
   GT     :  605  (99.51%)
   OTSU   :    3  (0.49%)

── Test-set provenance by class ──
col_0                      GT  OTSU
row_0                              
Dyskeratotic              122     0
Koilocytotic              123     1
Metaplastic               119     0
Parabasal                 118     0
Superficial-Intermediate  123     2


In [7]:

# ════════════════════════════════════════════════════════════
# CELL 6 — MORPHOLOGICAL PREPROCESSING (unified split)
# [C3] Imputer, IQR clip bounds and RobustScaler are now fit on the TRAINING
#      partition only and merely APPLIED to val/test. Fitting them on the full
#      matrix (as in v1) leaks test-set marginal statistics into the feature
#      transform and biases the reported morphological accuracy upward.
# ════════════════════════════════════════════════════════════

FEAT_COLS = [c for c in df_morph.columns
             if c not in ['Image', 'Class', 'seg_source']]  # [C6]

le_morph = LabelEncoder()
le_morph.fit(CANONICAL_CLASSES)
morph_class_names = list(le_morph.classes_)
assert morph_class_names == CANONICAL_CLASSES   # index space alignment

# Reorder the morph feature matrix into REGISTRY order, then slice by split.
X_reg_raw = df_morph[FEAT_COLS].values.astype(np.float64)[morph_rows]
X_reg_raw = np.where(np.isinf(X_reg_raw), np.nan, X_reg_raw)
y_reg     = y_unified.copy()

X_tr_raw = X_reg_raw[idx_train]

morph_imputer = SimpleImputer(strategy='median').fit(X_tr_raw)     # fit: train
_X_tr_imp     = morph_imputer.transform(X_tr_raw)
Q1, Q3        = np.percentile(_X_tr_imp, 25, axis=0), np.percentile(_X_tr_imp, 75, axis=0)
CLIP_LO, CLIP_HI = Q1 - 5*(Q3-Q1), Q3 + 5*(Q3-Q1)                  # bounds: train
morph_scaler  = RobustScaler().fit(np.clip(_X_tr_imp, CLIP_LO, CLIP_HI))

def morph_transform(X):
    """Apply the frozen train-fitted morphological transform."""
    X = np.where(np.isinf(X), np.nan, X)
    return morph_scaler.transform(np.clip(morph_imputer.transform(X), CLIP_LO, CLIP_HI))

X_morph_reg = morph_transform(X_reg_raw)

X_m_train, y_m_train = X_morph_reg[idx_train], y_reg[idx_train]
X_m_val,   y_m_val   = X_morph_reg[idx_val],   y_reg[idx_val]
X_m_test,  y_m_test  = X_morph_reg[idx_test],  y_reg[idx_test]

print(f'Morph  train={X_m_train.shape}  val={X_m_val.shape}  test={X_m_test.shape}')

Morph  train=(2833, 48)  val=(608, 48)  test=(608, 48)


In [8]:

# # ════════════════════════════════════════════════════════════
# # CELL 7 — MORPHOLOGICAL STACKED ENSEMBLE
# # ════════════════════════════════════════════════════════════

# base_learners = [
#     ('rf',  RandomForestClassifier(n_estimators=500, class_weight='balanced',
#                                    random_state=SEED, n_jobs=-1)),
#     ('et',  ExtraTreesClassifier(n_estimators=500,   class_weight='balanced',
#                                   random_state=SEED, n_jobs=-1)),
#     ('hgb', HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05,
#                                            max_depth=6, l2_regularization=0.1,
#                                            class_weight='balanced', random_state=SEED)),
#     ('mlp', MLPClassifier(hidden_layer_sizes=(256,128,64), alpha=0.01,
#                           max_iter=300, early_stopping=True, random_state=SEED)),
#     ('svc', CalibratedClassifierCV(
#         SVC(kernel='rbf', C=10.0, gamma='scale',
#             class_weight='balanced', random_state=SEED), cv=3))
# ]
# morph_stacker = StackingClassifier(
#     estimators=base_learners,
#     final_estimator=LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
#                                         random_state=SEED),
#     cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
#     stack_method='predict_proba', passthrough=True, n_jobs=-1
# )

# print('Training Morphological Stacked Ensemble...')
# morph_stacker.fit(X_m_train, y_m_train)

# # Probability tensors are kept as first-class artefacts — they are the inputs
# # to the meta-ensemble in Cell 13.
# P_morph_val  = morph_stacker.predict_proba(X_m_val)
# P_morph_test = morph_stacker.predict_proba(X_m_test)
# assert list(morph_stacker.classes_) == list(range(NUM_CLASSES)), \
#     'Stacker column order is not canonical — remap before fusing.'

# m_val_acc  = accuracy_score(y_m_val,  np.argmax(P_morph_val,  axis=1))
# m_test_acc = accuracy_score(y_m_test, np.argmax(P_morph_test, axis=1))
# print(f'\n① Morph Ensemble  Val={m_val_acc*100:.2f}%  Test={m_test_acc*100:.2f}%')
# print(classification_report(y_m_test, np.argmax(P_morph_test, axis=1),
#                              target_names=CANONICAL_CLASSES, zero_division=0))

In [9]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — MORPHOLOGICAL MODEL BENCHMARKING & DYNAMIC STACKING
# ════════════════════════════════════════════════════════════════════════════

print("--- Benchmarking 15 Standalone Models ---")

# 1. Define the suite of all algorithms
algorithms = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, class_weight='balanced', random_state=SEED),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=SEED, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, class_weight='balanced'),
    "CatBoost": cb.CatBoostClassifier(random_state=SEED, verbose=0, thread_count=-1, auto_class_weights='Balanced'),
    "SVC (RBF)": CalibratedClassifierCV(SVC(kernel='rbf', C=10.0, class_weight='balanced', random_state=SEED), cv=3),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=300, early_stopping=True, random_state=SEED),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
    "Gaussian Naive Bayes": GaussianNB(),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "AdaBoost": AdaBoostClassifier(random_state=SEED, algorithm='SAMME')
}

benchmark_results = []
trained_models = {}

# 2. Train and evaluate every model independently
for name, model in algorithms.items():
    start_time = time.time()
    
    # Train
    model.fit(X_m_train, y_m_train)
    trained_models[name] = model
    
    # Predict
    val_preds = model.predict(X_m_val)
    test_preds = model.predict(X_m_test)
    
    # Score
    val_acc = accuracy_score(y_m_val, val_preds)
    test_acc = accuracy_score(y_m_test, test_preds)
    test_bal = balanced_accuracy_score(y_m_test, test_preds)
    
    benchmark_results.append({
        "Model": name,
        "Val Accuracy": val_acc,
        "Test Accuracy": test_acc,
        "Test Balanced Acc": test_bal,
        "Train Time (s)": round(time.time() - start_time, 2)
    })
    
# 3. Display the sorted leaderboard
df_leaderboard = pd.DataFrame(benchmark_results).sort_values(by="Val Accuracy", ascending=False).reset_index(drop=True)
print("\n=== MODEL LEADERBOARD (Sorted by Validation Accuracy) ===")
display(df_leaderboard)

# 4. Dynamic Stacker Construction
# Change this integer to include more or fewer models, or hardcode a specific list of strings
TOP_N_MODELS = 5 
selected_model_names = df_leaderboard.head(TOP_N_MODELS)['Model'].tolist()

print(f"\n--- Constructing Stacker from Top {TOP_N_MODELS} Models ---")
print(f"Selected Base Learners: {selected_model_names}")

# Extract the trained parameter configurations for the selected models
base_learners = [(name, algorithms[name]) for name in selected_model_names]

morph_stacker = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=SEED),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    stack_method='predict_proba', passthrough=True, n_jobs=-1
)

print('Training Morphological Stacked Ensemble...')
morph_stacker.fit(X_m_train, y_m_train)

# 5. Final Evaluation for downstream branches
P_morph_val  = morph_stacker.predict_proba(X_m_val)
P_morph_test = morph_stacker.predict_proba(X_m_test)

assert list(morph_stacker.classes_) == list(range(NUM_CLASSES)), \
    'Stacker column order is not canonical — remap before fusing.'

m_val_acc  = accuracy_score(y_m_val,  np.argmax(P_morph_val,  axis=1))
m_test_acc = accuracy_score(y_m_test, np.argmax(P_morph_test, axis=1))
print(f'\n① Morph Ensemble Final Performance: Val={m_val_acc*100:.2f}% | Test={m_test_acc*100:.2f}%')
print(classification_report(y_m_test, np.argmax(P_morph_test, axis=1),
                             target_names=CANONICAL_CLASSES, zero_division=0))

--- Benchmarking 15 Standalone Models ---

=== MODEL LEADERBOARD (Sorted by Validation Accuracy) ===


,Model,Val Accuracy,Test Accuracy,Test Balanced Acc,Train Time (s)
0,LightGBM,0.957237,0.972039,0.972094,1.69
1,Hist Gradient Boosting,0.952303,0.968750,0.968867,3.24
2,XGBoost,0.952303,0.960526,0.960587,1.15
3,CatBoost,0.950658,0.965461,0.965316,24.72
4,SVC (RBF),0.939145,0.939145,0.939279,0.90
5,MLP (Neural Net),0.934211,0.940789,0.940948,1.66
6,Random Forest,0.932566,0.940789,0.940494,2.38
7,Extra Trees,0.932566,0.932566,0.932349,1.04
8,Logistic Regression,0.914474,0.940789,0.941040,0.12
9,LDA,0.904605,0.927632,0.927567,0.02



--- Constructing Stacker from Top 5 Models ---
Selected Base Learners: ['LightGBM', 'Hist Gradient Boosting', 'XGBoost', 'CatBoost', 'SVC (RBF)']
Training Morphological Stacked Ensemble...


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [15:52:50] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[15:53:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

[15:53:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

[15:53:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

[15:53:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

[15:53:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local


① Morph Ensemble Final Performance: Val=95.39% | Test=97.04%
                          precision    recall  f1-score   support

            Dyskeratotic       0.97      0.96      0.97       122
            Koilocytotic       0.94      0.94      0.94       124
             Metaplastic       0.96      0.98      0.97       119
               Parabasal       0.99      0.97      0.98       118
Superficial-Intermediate       0.99      0.99      0.99       125

                accuracy                           0.97       608
               macro avg       0.97      0.97      0.97       608
            weighted avg       0.97      0.97      0.97       608

